# Adversarial Robustness of a K-Means IDS — One-Hot Encoding Validation

This notebook rebuilds the K-means intrusion-detection experiment from the raw Zeek parquet data, but corrects the treatment of categorical features.

### Main change
Instead of representing `proto`, `conn_state`, `dest_port_zeek`, and `src_port_zeek` as ordered numbers inside the K-means distance calculation, they are treated as categorical variables and one-hot encoded.

### Goal
1. Load and clean the original data.
2. Bin source and destination ports using the same ranges as the previous experiment.
3. One-hot encode categorical features.
4. Standardize only the truly numeric features.
5. Train a new K-means model with `k=2`.
6. Identify the attack-dominant and normal-dominant clusters.
7. Find a real attack closest to the attack-cluster centroid.
8. Test realistic one-feature adversarial perturbations.
9. Check whether the previous protocol vulnerability persists after proper categorical encoding.


## 1. Imports


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when
from pyspark.sql.types import IntegerType, LongType, StructType, StructField, StringType

from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import (
    VectorAssembler,
    StandardScaler,
    StringIndexer,
    OneHotEncoder
)
from pyspark.ml.functions import vector_to_array

import numpy as np
import pandas as pd

from functools import reduce


## 2. Start Spark

This uses the same general Spark setup as the original notebook.


In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("K-means Research - One Hot Encoding")
    .config("spark.ui.port", "4050")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")


26/09/30 21:20:06 WARN Utils: Your hostname, car-ubuntu resolves to a loopback address: 127.0.1.1; using 192.168.1.42 instead (on interface ens33)
26/09/30 21:20:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 21:20:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/30 21:20:08 WARN Utils: Service 'SparkUI' could not bind on port 4050. Attempting port 4051.
26/09/30 21:20:08 WARN Utils: Service 'SparkUI' could not bind on port 4051. Attempting port 4052.
26/09/30 21:20:08 WARN Utils: Service 'SparkUI' could not bind on port 4052. Attempting port 4053.


## 3. Load the Zeek parquet dataset

This uses the same dataset path as the previous notebook.

If the folder is moved later, change only `file_path`.


In [3]:
file_path = "/home/ubuntu/Downloads/Zeek_Parquet"

df = spark.read.parquet(file_path)

print("Total records:", df.count())
print("Total columns:", len(df.columns))


Total records: 700340
Total columns: 25


## 4. Basic dataset checks

The model is unsupervised, so `label_binary` is **not** used to train K-means.  
It is used afterward to determine which cluster is attack-dominant and which is normal-dominant.

This cell also checks whether any rows have a missing label.


In [4]:
attack_count = df.filter(F.col("label_binary") == True).count()
normal_count = df.filter(F.col("label_binary") == False).count()
null_label_count = df.filter(F.col("label_binary").isNull()).count()

print("Attack labels:", attack_count)
print("Normal labels:", normal_count)
print("Null labels:", null_label_count)
print("Total:", df.count())


Attack labels: 346933
Normal labels: 350339
Null labels: 0
Total: 700340


In [5]:
df.select(
    "duration",
    "proto",
    "conn_state",
    "orig_bytes",
    "orig_ip_bytes",
    "orig_pkts",
    "resp_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "missed_bytes",
    "dest_port_zeek",
    "src_port_zeek",
    "label_binary"
).limit(5).toPandas()


,duration,proto,conn_state,orig_bytes,orig_ip_bytes,orig_pkts,resp_bytes,resp_pkts,resp_ip_bytes,missed_bytes,dest_port_zeek,src_port_zeek,label_binary
0,NaN,tcp,S0,NaN,44,1,NaN,0,0,0,36197,44183,True
1,NaN,tcp,S0,NaN,44,1,NaN,0,0,0,8323,44184,True
2,NaN,tcp,S0,NaN,44,1,NaN,0,0,0,47158,44183,True
3,NaN,tcp,S0,NaN,44,1,NaN,0,0,0,34135,44183,True
4,NaN,tcp,S0,NaN,44,1,NaN,0,0,0,9838,44184,True


## 5. Define the model features

The experiment keeps the same 12 conceptual features as the earlier K-means model.

### Numeric features
These represent quantities and will be standardized.

### Categorical features
These represent categories and will be one-hot encoded.

Importantly, `proto` and `conn_state` are **not manually converted to 0, 1, 2...** in this notebook.


In [6]:
columns_to_check = [
    "duration",
    "resp_pkts",
    "conn_state",
    "proto",
    "orig_ip_bytes",
    "missed_bytes",
    "orig_pkts",
    "resp_ip_bytes",
    "dest_port_zeek",
    "orig_bytes",
    "resp_bytes",
    "src_port_zeek"
]

categorical_features = [
    "conn_state",
    "proto",
    "dest_port_zeek",
    "src_port_zeek"
]

numeric_features = [
    feature
    for feature in columns_to_check
    if feature not in categorical_features
]

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)


Categorical features: ['conn_state', 'proto', 'dest_port_zeek', 'src_port_zeek']
Numeric features: ['duration', 'resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'orig_pkts', 'resp_ip_bytes', 'orig_bytes', 'resp_bytes']


## 6. Clean missing values

Numeric missing values are replaced with 0, matching the earlier preprocessing logic.

For the string categorical variables (`proto` and `conn_state`), missing values are represented explicitly as `"unknown"` rather than assigning them an arbitrary number.


In [7]:
df_clean = df.na.fill(
    0,
    subset=numeric_features
)

df_clean = df_clean.na.fill(
    {
        "proto": "unknown",
        "conn_state": "unknown"
    }
)


## 7. Bin source and destination ports

Use the same bins as the previous experiment:

- **0** = missing / null
- **1** = well-known ports, 0–1023
- **2** = registered ports, 1024–49151
- **3** = dynamic/private ports, 49152–65535
- **4** = values outside the expected port range

The resulting bin numbers are treated as **categories** later, not as continuous numeric distances.


In [8]:
def bin_ports(dataframe, column):
    return dataframe.withColumn(
        column,
        when(col(column).isNull(), 0)
        .when((col(column) >= 0) & (col(column) <= 1023), 1)
        .when((col(column) >= 1024) & (col(column) <= 49151), 2)
        .when((col(column) >= 49152) & (col(column) <= 65535), 3)
        .otherwise(4)
    )

for port_column in ["dest_port_zeek", "src_port_zeek"]:
    df_clean = bin_ports(df_clean, port_column)


In [9]:
print("Protocols:")
df_clean.groupBy("proto").count().orderBy(F.desc("count")).show()

print("Connection states:")
df_clean.groupBy("conn_state").count().orderBy(F.desc("count")).show()

print("Destination port bins:")
df_clean.groupBy("dest_port_zeek").count().orderBy("dest_port_zeek").show()

print("Source port bins:")
df_clean.groupBy("src_port_zeek").count().orderBy("src_port_zeek").show()


Protocols:
+-----+------+
|proto| count|
+-----+------+
|  udp|388470|
|  tcp|310910|
| icmp|   960|
+-----+------+

Connection states:
+----------+------+
|conn_state| count|
+----------+------+
|        S0|587615|
|        SF| 75749|
|        SH| 20682|
|       OTH|  9897|
|       SHR|  5249|
|     RSTRH|   338|
|       REJ|   306|
|      RSTR|   300|
|      RSTO|   158|
|        S1|    39|
|        S2|     7|
+----------+------+

Destination port bins:
+--------------+------+
|dest_port_zeek| count|
+--------------+------+
|             1|397334|
|             2|232155|
|             3| 70851|
+--------------+------+

Source port bins:
+-------------+------+
|src_port_zeek| count|
+-------------+------+
|            1|  7558|
|            2|373588|
|            3|319194|
+-------------+------+



# One-Hot Encoding Pipeline

The following pipeline creates a corrected feature representation for K-means.

### Why `dropLast=False`?
For K-means, we do not want one category to become the special all-zero reference category. Keeping every one-hot dimension makes pairwise categorical differences symmetric.


## 8. Create categorical indexers


In [10]:
indexers_ohe = [
    StringIndexer(
        inputCol=feature,
        outputCol=f"{feature}_idx",
        handleInvalid="keep"
    )
    for feature in categorical_features
]


## 9. One-hot encode the categorical variables


In [11]:
indexed_cols_ohe = [
    f"{feature}_idx"
    for feature in categorical_features
]

encoded_cols_ohe = [
    f"{feature}_vec"
    for feature in categorical_features
]

encoder_ohe = OneHotEncoder(
    inputCols=indexed_cols_ohe,
    outputCols=encoded_cols_ohe,
    handleInvalid="keep",
    dropLast=False
)


## 10. Assemble and standardize only the numeric features

The categorical one-hot vectors are **not** standardized.  
The numeric variables are standardized so differences in units do not dominate Euclidean distance.


In [12]:
numeric_assembler_ohe = VectorAssembler(
    inputCols=numeric_features,
    outputCol="numeric_features_ohe"
)

numeric_scaler_ohe = StandardScaler(
    inputCol="numeric_features_ohe",
    outputCol="scaled_numeric_ohe",
    withStd=True,
    withMean=False
)


## 11. Combine scaled numeric features with one-hot categorical vectors


In [13]:
final_assembler_ohe = VectorAssembler(
    inputCols=[
        "scaled_numeric_ohe",
        *encoded_cols_ohe
    ],
    outputCol="features_ohe"
)


## 12. Fit the preprocessing pipeline

The fitted preprocessing model must later be reused for all adversarial samples.  
Do **not** fit a new scaler or encoder on adversarial records.


In [14]:
preprocess_ohe = Pipeline(
    stages=[
        *indexers_ohe,
        encoder_ohe,
        numeric_assembler_ohe,
        numeric_scaler_ohe,
        final_assembler_ohe
    ]
)

preprocess_ohe_model = preprocess_ohe.fit(df_clean)

df_ohe = preprocess_ohe_model.transform(df_clean)


## 13. Check the new feature-vector size

The new vector should have more than 12 dimensions because each categorical variable is now represented by multiple one-hot dimensions.


In [15]:
example_vector = df_ohe.select("features_ohe").first()["features_ohe"]

print("Old conceptual feature count:", len(columns_to_check))
print("New one-hot vector dimensions:", example_vector.size)


Old conceptual feature count: 12
New one-hot vector dimensions: 36


# Train the New K-Means Model

Use `k=2` again so this validation remains directly comparable with the earlier experiment.


## 14. Train K-means


In [16]:
kmeans_ohe = KMeans(
    featuresCol="features_ohe",
    predictionCol="cluster_ohe",
    k=2,
    seed=200
)

model_ohe = kmeans_ohe.fit(df_ohe)

df_pred_ohe = model_ohe.transform(df_ohe)


26/09/30 21:21:46 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

## 15. Evaluate clustering compactness

Silhouette is not a supervised IDS accuracy measure.  
It only gives information about how well separated the clusters are geometrically.


In [17]:
evaluator_ohe = ClusteringEvaluator(
    featuresCol="features_ohe",
    predictionCol="cluster_ohe",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette_ohe = evaluator_ohe.evaluate(df_pred_ohe)

print("Silhouette score:", silhouette_ohe)


[Stage 65:===========================================>              (3 + 1) / 4]

Silhouette score: 0.9999840565900945


## 16. Determine which cluster is attack-dominant

K-means cluster numbers are arbitrary.  
Do **not** assume cluster 1 is still the attack cluster.

The true labels are used only here, after clustering, to characterize the two clusters.


In [18]:
cluster_stats_ohe = (
    df_pred_ohe
    .groupBy("cluster_ohe")
    .agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(F.col("label_binary") == True, 1).otherwise(0)
        ).alias("attacks"),
        F.sum(
            F.when(F.col("label_binary") == False, 1).otherwise(0)
        ).alias("normal")
    )
    .withColumn(
        "labeled_total",
        F.col("attacks") + F.col("normal")
    )
    .withColumn(
        "attack_rate",
        F.when(
            F.col("labeled_total") > 0,
            F.col("attacks") / F.col("labeled_total")
        )
    )
)

cluster_stats_ohe.orderBy("cluster_ohe").show(truncate=False)


[Stage 68:===========================================>              (3 + 1) / 4]

+-----------+------+-------+------+-------------+------------------+
|cluster_ohe|total |attacks|normal|labeled_total|attack_rate       |
+-----------+------+-------+------+-------------+------------------+
|0          |700331|346933 |350330|697263       |0.4975640468517618|
|1          |9     |0      |9     |9            |0.0               |
+-----------+------+-------+------+-------------+------------------+



In [19]:
attack_cluster_ohe = (
    cluster_stats_ohe
    .orderBy(F.desc("attack_rate"))
    .first()["cluster_ohe"]
)

normal_cluster_ohe = (
    cluster_stats_ohe
    .orderBy(F.asc("attack_rate"))
    .first()["cluster_ohe"]
)

print("Attack-dominant cluster:", attack_cluster_ohe)
print("Normal-dominant cluster:", normal_cluster_ohe)


[Stage 74:===========================================>              (3 + 1) / 4]

Attack-dominant cluster: 0
Normal-dominant cluster: 1


# Find a Representative Real Attack

Because the feature geometry changed, the representative attack must be selected again.

We find the **true malicious record inside the attack-dominant cluster that is closest to that cluster's centroid**.


## 17. Get the attack centroid


In [20]:
centers_ohe = model_ohe.clusterCenters()

attack_center_ohe = np.array(
    centers_ohe[attack_cluster_ohe],
    dtype=float
)

print("Feature dimensions:", len(attack_center_ohe))


Feature dimensions: 36


## 18. Calculate each true attack's distance to the attack centroid


In [21]:
attack_records_ohe = (
    df_pred_ohe
    .filter(
        (F.col("cluster_ohe") == attack_cluster_ohe) &
        (F.col("label_binary") == True)
    )
    .withColumn(
        "_features_array_ohe",
        vector_to_array("features_ohe")
    )
)

distance_terms_ohe = [
    F.pow(
        F.col("_features_array_ohe")[i] -
        F.lit(float(attack_center_ohe[i])),
        2
    )
    for i in range(len(attack_center_ohe))
]

distance_ohe = reduce(
    lambda a, b: a + b,
    distance_terms_ohe
)

attack_records_ohe = attack_records_ohe.withColumn(
    "distance_to_attack_centroid_ohe",
    distance_ohe
)


## 19. Select the closest real attack and store its raw feature values


In [22]:
closest_attack_ohe = (
    attack_records_ohe
    .orderBy("distance_to_attack_centroid_ohe")
    .first()
)

baseline_ohe = {
    feature: closest_attack_ohe[feature]
    for feature in columns_to_check
}

print(
    "Squared distance to attack centroid:",
    closest_attack_ohe["distance_to_attack_centroid_ohe"]
)

print("\nRepresentative attack baseline:")
print(pd.DataFrame([baseline_ohe]).to_string(index=False))


26/09/30 21:22:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 77:===========================================>              (3 + 1) / 4]

Squared distance to attack centroid: 1.170343607250852

Representative attack baseline:
 duration  resp_pkts conn_state proto  orig_ip_bytes  missed_bytes  orig_pkts  resp_ip_bytes  dest_port_zeek  orig_bytes  resp_bytes  src_port_zeek
 0.000037          0         S0   udp            140             0          2              0               1          84           0              2


## 20. Verify that the selected baseline is assigned to the attack cluster

This must pass before adversarial testing.


In [23]:
baseline_ohe_df = (
    spark.createDataFrame([baseline_ohe])
    .select(*columns_to_check)
)

baseline_ohe_processed = preprocess_ohe_model.transform(
    baseline_ohe_df
)

baseline_ohe_prediction = (
    model_ohe
    .transform(baseline_ohe_processed)
    .select("cluster_ohe")
    .first()[0]
)

print("Baseline cluster:", baseline_ohe_prediction)
print("Attack-dominant cluster:", attack_cluster_ohe)

assert baseline_ohe_prediction == attack_cluster_ohe, (
    "The selected baseline was unexpectedly not assigned to the attack cluster."
)


Baseline cluster: 0
Attack-dominant cluster: 0


# Constrained Adversarial Robustness Test

Now repeat the realistic single-feature experiment.

The question is:

> **Can changing only one attacker-controllable feature, while keeping every other feature identical, move the representative malicious flow into the normal-dominant cluster?**


## 21. Define attacker-controllable features

These match the previous adversarial experiment.


In [24]:
features_to_change_ohe = [
    "duration",
    "conn_state",
    "proto",
    "orig_ip_bytes",
    "orig_pkts",
    "dest_port_zeek",
    "orig_bytes",
    "src_port_zeek"
]

categorical_attack_ohe = [
    "conn_state",
    "proto",
    "dest_port_zeek",
    "src_port_zeek"
]

numeric_attack_ohe = [
    "duration",
    "orig_ip_bytes",
    "orig_pkts",
    "orig_bytes"
]

features_to_keep_fixed_ohe = [
    feature
    for feature in columns_to_check
    if feature not in features_to_change_ohe
]

print("Features allowed to change:", features_to_change_ohe)
print("Features kept fixed:", features_to_keep_fixed_ohe)


Features allowed to change: ['duration', 'conn_state', 'proto', 'orig_ip_bytes', 'orig_pkts', 'dest_port_zeek', 'orig_bytes', 'src_port_zeek']
Features kept fixed: ['resp_pkts', 'missed_bytes', 'resp_ip_bytes', 'resp_bytes']


## 22. Use real attacks to define plausible candidate values

Categorical candidates are values that actually occur among true malicious records.

Numeric candidates will use the 5th, 25th, 50th, 75th and 95th percentiles of true attacks, plus the baseline value.


In [25]:
attack_data_ohe = df_clean.filter(
    F.col("label_binary") == True
)

raw_feature_schema_ohe = df_clean.select(
    *columns_to_check
).schema

def cast_like_original_ohe(feature, value):
    dtype = raw_feature_schema_ohe[feature].dataType

    if isinstance(dtype, (IntegerType, LongType)):
        return int(round(value))

    return float(value)


## 23. Build realistic categorical candidate values


In [26]:
realistic_grid_ohe = {}

for feature in categorical_attack_ohe:
    realistic_grid_ohe[feature] = [
        row[feature]
        for row in (
            attack_data_ohe
            .select(feature)
            .distinct()
            .orderBy(feature)
            .collect()
        )
    ]


## 24. Build realistic numeric candidate values

Using percentiles avoids the extreme maximum values that dominated the earlier broad Cartesian search.


In [27]:
realistic_quantiles_ohe = [
    0.05,
    0.25,
    0.50,
    0.75,
    0.95
]

for feature in numeric_attack_ohe:

    values = attack_data_ohe.approxQuantile(
        feature,
        realistic_quantiles_ohe,
        0.001
    )

    values.append(baseline_ohe[feature])

    values = [
        cast_like_original_ohe(feature, value)
        for value in values
    ]

    realistic_grid_ohe[feature] = sorted(set(values))


## 25. Inspect the realistic search grid

Review this output before continuing.  
The candidate values should be plausible and should not contain the extreme outliers that drove the original broad search.


In [28]:
for feature, values in realistic_grid_ohe.items():
    print(
        feature,
        "| baseline =", baseline_ohe[feature],
        "| tested =", values
    )


conn_state | baseline = S0 | tested = ['OTH', 'REJ', 'RSTO', 'RSTR', 'RSTRH', 'S0', 'S1', 'S2', 'SF', 'SH']
proto | baseline = udp | tested = ['tcp', 'udp']
dest_port_zeek | baseline = 1 | tested = [1, 2, 3]
src_port_zeek | baseline = 2 | tested = [1, 2, 3]
duration | baseline = 3.695487976074219e-05 | tested = [0.0, 3.695487976074219e-05, 0.0006241798400878906]
orig_ip_bytes | baseline = 140 | tested = [40, 44, 140, 195]
orig_pkts | baseline = 2 | tested = [1, 2, 3]
orig_bytes | baseline = 84 | tested = [0, 84, 114]


# Single-Feature Adversarial Test

Every generated record differs from the representative attack in **exactly one feature**.


## 26. Generate one-feature perturbations


In [29]:
single_records_ohe = []

for feature in features_to_change_ohe:

    for value in realistic_grid_ohe[feature]:

        if value == baseline_ohe[feature]:
            continue

        record = baseline_ohe.copy()

        record[feature] = value
        record["changed_feature"] = feature
        record["changed_value"] = str(value)

        single_records_ohe.append(record)

print("Single-feature experiments:", len(single_records_ohe))


Single-feature experiments: 23


## 27. Create a Spark DataFrame for the adversarial records


In [30]:
single_schema_ohe = StructType(
    raw_feature_schema_ohe.fields + [
        StructField(
            "changed_feature",
            StringType(),
            False
        ),
        StructField(
            "changed_value",
            StringType(),
            False
        )
    ]
)

single_df_ohe = spark.createDataFrame(
    single_records_ohe,
    schema=single_schema_ohe
)


## 28. Apply the exact preprocessing learned from the original dataset

This is critical.

The encoder and scaler are **not fitted again** on adversarial records.  
The already-fitted `preprocess_ohe_model` is reused.


In [31]:
single_processed_ohe = preprocess_ohe_model.transform(
    single_df_ohe
)

single_predictions_ohe = model_ohe.transform(
    single_processed_ohe
)


## 29. Mark successful evasions

A successful evasion occurs when a record that originated from the representative malicious baseline is reassigned to the **normal-dominant cluster**.


In [32]:
single_results_ohe = single_predictions_ohe.withColumn(
    "evasion",
    F.when(
        F.col("cluster_ohe") == normal_cluster_ohe,
        1
    ).otherwise(0)
)


## 30. Show every single-feature result


In [33]:
single_results_ohe.select(
    "changed_feature",
    "changed_value",
    "cluster_ohe",
    "evasion"
).show(
    100,
    truncate=False
)


+---------------+---------------------+-----------+-------+
|changed_feature|changed_value        |cluster_ohe|evasion|
+---------------+---------------------+-----------+-------+
|duration       |0.0                  |0          |0      |
|duration       |0.0006241798400878906|0          |0      |
|conn_state     |OTH                  |0          |0      |
|conn_state     |REJ                  |0          |0      |
|conn_state     |RSTO                 |0          |0      |
|conn_state     |RSTR                 |0          |0      |
|conn_state     |RSTRH                |0          |0      |
|conn_state     |S1                   |0          |0      |
|conn_state     |S2                   |0          |0      |
|conn_state     |SF                   |0          |0      |
|conn_state     |SH                   |0          |0      |
|proto          |tcp                  |0          |0      |
|orig_ip_bytes  |40                   |0          |0      |
|orig_ip_bytes  |44                   |0

## 31. Show only successful single-feature evasions


In [34]:
successful_single_ohe = single_results_ohe.filter(
    F.col("evasion") == 1
)

print(
    "Successful single-feature evasions:",
    successful_single_ohe.count()
)

successful_single_ohe.select(
    "changed_feature",
    "changed_value",
    *features_to_change_ohe,
    "cluster_ohe"
).show(
    100,
    truncate=False
)


Successful single-feature evasions: 0
+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+-----------+
|changed_feature|changed_value|duration|conn_state|proto|orig_ip_bytes|orig_pkts|dest_port_zeek|orig_bytes|src_port_zeek|cluster_ohe|
+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+-----------+
+---------------+-------------+--------+----------+-----+-------------+---------+--------------+----------+-------------+-----------+



## 32. Summarize vulnerability by feature


In [35]:
single_summary_ohe = (
    single_results_ohe
    .groupBy("changed_feature")
    .agg(
        F.count("*").alias("tested"),
        F.sum("evasion").alias("successful"),
        F.avg("evasion").alias("evasion_rate")
    )
    .orderBy(F.desc("evasion_rate"))
)

single_summary_ohe.show(
    truncate=False
)


+---------------+------+----------+------------+
|changed_feature|tested|successful|evasion_rate|
+---------------+------+----------+------------+
|duration       |2     |0         |0.0         |
|conn_state     |9     |0         |0.0         |
|orig_ip_bytes  |3     |0         |0.0         |
|proto          |1     |0         |0.0         |
|orig_pkts      |2     |0         |0.0         |
|orig_bytes     |2     |0         |0.0         |
|src_port_zeek  |2     |0         |0.0         |
|dest_port_zeek |2     |0         |0.0         |
+---------------+------+----------+------------+



## 33. Inspect protocol changes specifically

This is the key validation of the earlier finding.

In this notebook, `proto` remains in its original string form (for example `"tcp"` and `"udp"`), so the result is easier to interpret than the previous 0/1/2 encoding.


In [36]:
single_results_ohe.filter(
    F.col("changed_feature") == "proto"
).select(
    "changed_feature",
    "changed_value",
    "cluster_ohe",
    "evasion"
).show(
    truncate=False
)


+---------------+-------------+-----------+-------+
|changed_feature|changed_value|cluster_ohe|evasion|
+---------------+-------------+-----------+-------+
|proto          |tcp          |0          |0      |
+---------------+-------------+-----------+-------+



# Stop Here and Interpret the Results

The most important outputs to save are:

1. **Cluster statistics** from Section 16.
2. **The new representative baseline attack** from Section 19.
3. **All single-feature results** from Section 30.
4. **Successful evasions** from Section 31.
5. **Feature-level summary** from Section 32.
6. **Protocol-specific results** from Section 33.

### Main validation question

Does a realistic change such as `tcp → udp` still move the representative malicious flow into the normal-dominant cluster after categorical features are properly one-hot encoded?

- If **yes**, the original adversarial result becomes substantially stronger.
- If **no**, the earlier ordinal encoding was likely contributing to the vulnerability, which is itself an important methodological result.

After interpreting these outputs, the next experiment should repeat the two-feature adversarial test and then test generalization across many real malicious records.
